<a href="https://colab.research.google.com/github/saimamanzoor651/code-switching-codesaviours-si26-saima/blob/main/SI26_Week7_Saima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import re


URD_WORDS = set("""
aaj kal din raat subah shaam mera meri mere tera teri tere uska uski uske
hai hain tha thi the ka ki ke ko se me main mein tum ap aap yaar bhai behan
yar dost kya kyun kyu kaise kahan kab kaun kitna kitni bohot bahut zyada kam
thora thori acha achi accha bura buri nahi nahin haan ji zaroor shayad
shukriya meherbani please_urd wapis wapas phir dobara abhi ab tak se pehle
baad khana pani sona jagna jana ana lena dena karna hona rehna dekhna
sunna bolna kehna likhna parhna samajhna bhoolna yaad milna dena
lagna banana khelna kaam ghar bahar andar bazar dukaan gari school
college university ammi abbu bhai behn dost sath akela tanha khush
udaas pareshan thak thaka thaki gaya gayi gaye raha rahi rahe karo
karain karenge hoga hogi honge dega degi denge lo lena lelo dekha
dekhi socha sochti sochta mila mili bola boli kaha kahi likha likhi
mubarak dua meherban maaf maafi sorry_urd theek thik acha_wala waisay
waise bas itna itni sirf lekin magar phir_bhi agar warna kyunke kyunki
tou to hi bhi na nahi_hai kuch kuchh sab sabko har koi kisi kisiko
mujhe tumhe usay hume ise unhe apna apni apne wala wali wale
""".split())

raw_entries = []

def add(sentence, tags):
    raw_entries.append((sentence, tags))

sentences_with_labels = [
    ("Aaj ka din bohot busy tha", ["URD","URD","URD","URD","ENG","URD"]),
    ("Yaar seriously I can not even right now", ["URD","ENG","ENG","ENG","ENG","ENG","ENG","ENG"]),
    ("Bhai kal mera presentation hai still not prepared at all", ["URD","URD","URD","ENG","URD","ENG","ENG","ENG","ENG","ENG"]),
    ("Khana kha liya I was literally waiting for you", ["URD","URD","URD","ENG","ENG","ENG","ENG","ENG","ENG"]),
    ("Research team ne core data analysis publish ki", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("New mobile release first week benchmark touch kiya", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("City traffic control team emergency operational guide issue", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG"]),
    ("Student portal login error fix solve ho gaya", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Project timeline phase two start hone wala hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Is software release note me technical detail include", ["URD", "ENG", "ENG", "ENG", "URD", "ENG", "ENG", "ENG"]),
    ("Local event management team safety protocol follow kare", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD"]),

    ("Mera storage memory full alert receive hone laga", ["URD", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Is laptop screen bright display balance sahi hai", ["URD", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("New client onboarding process timing schedule ban gaya", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Mujhe abhi ek meeting attend karni hai", ["URD","URD","URD","ENG","ENG","URD","URD"]),
    ("Yeh assignment submit karna bhool gaya tha", ["URD","ENG","ENG","URD","URD","URD","URD"]),
    ("Please jaldi reply kar do na", ["MIX","URD","ENG","URD","URD","URD"]),
    ("Mera mood off hai aaj kal", ["URD","URD","ENG","URD","URD","URD"]),
    ("Weekend pe koi plan hai kya", ["ENG","URD","URD","ENG","URD","URD"]),

    ("Traffic itna zyada tha ke late ho gaya", ["ENG","URD","URD","URD","URD","ENG","URD","URD"]),
    ("Exam ki tayari start kar do ab", ["ENG","URD","URD","ENG","URD","URD","URD"]),
    ("Bilkul theek hai I will handle it", ["URD","URD","URD","ENG","ENG","ENG","ENG"]),
    ("Kal office mein ek important call hai", ["URD","ENG","URD","URD","ENG","ENG","URD"]),
    ("Mujhe iska koi idea nahi tha", ["URD","URD","URD","ENG","URD","URD"]),
    ("Zara wait karo main aa raha hoon", ["URD","ENG","URD","URD","URD","URD","URD"]),
    ("Yeh project deadline bohot close hai", ["URD","ENG","ENG","URD","ENG","URD"]),
    ("Mummy ne kaha ke ghar jaldi aao", ["URD","URD","URD","URD","URD","URD","URD"]),
    ("Honestly mujhe iska matlab samajh nahi aya", ["ENG","URD","URD","URD","URD","URD","URD"]),
    ("Chalo chalte hain warna late ho jayenge", ["URD","URD","URD","URD","ENG","URD","URD"]),
    ("Mera phone ki battery low ho gayi hai", ["URD","ENG","URD","ENG","ENG","URD","URD","URD"]),
    ("Uski shaadi next month hai", ["URD","URD","ENG","ENG","URD"]),
    ("Is video ko like aur share zaroor karna", ["URD","ENG","URD","ENG","URD","ENG","URD","URD"]),
    ("Bhookh lagi hai kuch order karte hain", ["URD","URD","URD","URD","ENG","URD","URD"]),
    ("Mujhe wo movie bohot pasand ayi thi", ["URD","URD","ENG","URD","URD","URD","URD"]),
    ("Class mein aaj attendance nahi hui", ["ENG","URD","URD","ENG","URD","URD"]),

    ("Bas ek second ruko main check karta hoon", ["URD","URD","ENG","URD","URD","ENG","URD","URD"]),
    ("Yeh internet connection bohot slow chal raha hai", ["URD","ENG","ENG","URD","ENG","URD","URD","URD"]),
    ("Kaam se thak ke ghar aya hoon", ["URD","URD","URD","URD","URD","URD","URD"]),
    ("I think humein plan change karna chahiye", ["ENG","ENG","URD","ENG","ENG","URD","URD"]),
    ("Subah jaldi uthna bohot mushkil hai", ["URD","URD","URD","URD","URD","URD"]),
    ("Mera laptop kharab ho gaya hai suddenly", ["URD","ENG","URD","URD","URD","URD","ENG"]),
    ("Ammi ne biryani banai hai aaj", ["URD","URD","URD","URD","URD","URD"]),
    ("Yaar tumhe pata hai kal weather kaisa hoga", ["URD","URD","URD","URD","URD","ENG","URD","URD"]),
    ("Office ka kaam khatam nahi ho raha", ["ENG","URD","URD","URD","URD","URD","URD"]),
    ("Chai peene chalein warna neend a rahi hai", ["URD","URD","URD","URD","URD","URD","URD","URD"]),
    ("Mujhe lagta hai ye decision wrong hai", ["URD","URD","URD","URD","ENG","ENG","URD"]),
    ("Kal se mera diet start ho raha hai", ["URD","URD","URD","ENG","URD","URD","URD","URD"]),
    ("Group project ka kaam divide kar lete hain", ["ENG","ENG","URD","URD","ENG","URD","URD","URD"]),
    ("Mera dil nahi kar raha kuch karne ka", ["URD","URD","URD","URD","URD","URD","URD","URD"]),
    ("Abhi tak reply nahi aya unka", ["URD","URD","ENG","URD","URD","URD"]),

    ("Bhai wo match kitna exciting tha", ["URD","URD","ENG","URD","ENG","URD"]),
    ("Mujhe thora tension ho raha hai exam ka", ["URD","URD","ENG","URD","URD","URD","ENG","URD"]),
    ("Chalo movie dekhne chalte hain tonight", ["URD","ENG","URD","URD","URD","ENG"]),
    ("Yeh update install karne mein time lagega", ["URD","ENG","URD","URD","URD","ENG","URD"]),
    ("Mummy papa aj shaam ko ayenge", ["URD","URD","URD","URD","URD","URD"]),
    ("I am so tired yaar bilkul thak gaya", ["ENG","ENG","ENG","ENG","URD","URD","URD","URD"]),
    ("Kal ka lecture bohot boring tha honestly", ["URD","URD","ENG","URD","ENG","URD","MIX"]),
    ("Mujhe ye job offer accept karna chahiye ya nahi", ["URD","URD","ENG","ENG","ENG","URD","URD","URD","URD"]),
    ("Bhai signal nahi a raha phone pe", ["URD","ENG","URD","URD","URD","ENG","URD"]),
    ("Ghar pohanchte hi message kar dena", ["URD","URD","URD","ENG","URD","URD"]),
    ("Aj ka mausam bohot pleasant hai", ["URD","URD","URD","URD","ENG","URD"]),
    ("Yaar mujhe koi motivation nahi mil raha", ["URD","URD","URD","ENG","URD","URD","URD"]),
    ("Kal ki meeting postpone ho gayi hai", ["URD","URD","ENG","ENG","URD","URD","URD"]),
    ("Mera result kal announce hoga", ["URD","URD","URD","ENG","URD"]),
    ("Bus miss ho gayi mujhse", ["ENG","ENG","URD","URD","URD"]),

    ("Yeh coffee bohot strong bani hai", ["URD","ENG","URD","ENG","URD","URD"]),
    ("Kya tum free ho abhi", ["URD","URD","ENG","URD","URD"]),
    ("Mujhe ye news sunke shock laga", ["URD","URD","ENG","URD","ENG","URD"]),
    ("Sab kuch plan ke mutabiq ho raha hai", ["URD","URD","ENG","URD","URD","URD","URD","URD"]),
    ("Weekend pe ghar clean karna hai poora", ["ENG","URD","URD","ENG","URD","URD","URD"]),
    ("Mera code compile nahi ho raha kal se", ["URD","ENG","ENG","URD","URD","URD","URD","URD"]),
    ("Itni garmi hai bahar nikalne ka dil nahi karta", ["URD","URD","URD","URD","URD","URD","URD","URD","URD"]),
    ("Bhai yeh app bohot slow chal rahi hai", ["URD","URD","ENG","URD","ENG","URD","URD","URD"]),
    ("Mujhe office se late reply mila", ["URD","ENG","URD","ENG","ENG","URD"]),
    ("Kal exam hai stress bohot ho raha hai", ["URD","ENG","URD","ENG","URD","URD","URD","URD"]),
    ("Yaar mera wifi ka router hang ho gaya hai", ["URD","URD","ENG","URD","ENG","ENG","URD","URD","URD"]),
    ("Subah se sar dard ho raha hai bohot", ["URD","URD","URD","URD","URD","URD","URD","URD"]),
    ("Mera flight cancel ho gayi thi last week", ["URD","ENG","ENG","URD","URD","URD","ENG","ENG"]),
    ("Ammi ne bola ke jaldi so jao", ["URD","URD","URD","URD","URD","URD","URD"]),

    ("Yeh outfit kaisa lag raha hai mujhpe", ["URD","ENG","URD","URD","URD","URD","URD"]),
    ("Mujhe apna schedule manage karna mushkil lag raha hai", ["URD","URD","ENG","ENG","URD","URD","URD","URD","URD"]),
    ("Kal se gym start kar raha hoon finally", ["URD","URD","ENG","ENG","URD","URD","URD","ENG"]),
    ("Yaar ye internship offer letter aya hai", ["URD","URD","ENG","ENG","ENG","URD","URD"]),
    ("Mera mind kuch aur soch raha hai abhi", ["URD","ENG","URD","URD","URD","URD","URD","URD"]),
    ("Rasta bohot jam tha isliye late ho gaya", ["URD","URD","URD","URD","URD","ENG","URD","URD"]),
    ("Yeh recipe try karni hai weekend pe", ["URD","ENG","ENG","URD","URD","ENG","URD"]),
    ("Mera roommate bohot messy hai yaar", ["URD","ENG","URD","ENG","URD","URD"]),
    ("Aj ka din kaafi productive raha honestly", ["URD","URD","URD","URD","ENG","URD","MIX"]),
    ("Mujhe iska solution samajh nahi araha", ["URD","URD","ENG","URD","URD","URD"]),
    ("Kal ka paper kaisa gaya tumhara", ["URD","URD","ENG","URD","URD","URD"]),

    ("Yaar wo restaurant ka khana bohot tasty tha", ["URD","URD","ENG","URD","URD","URD","ENG","URD"]),
    ("Mera plan hai ke kal early sona hai", ["URD","ENG","URD","URD","URD","ENG","URD","URD"]),
    ("Kya tumne homework complete kar liya", ["URD","URD","ENG","ENG","URD","URD"]),
    ("Bhai mujhe ek favor chahiye tumse", ["URD","URD","URD","ENG","URD","URD"]),
    ("Yeh dress mujhpe kaisi lagegi", ["URD","ENG","URD","URD","URD"]),
    ("Mujhe office jaate waqt traffic mil gaya", ["URD","ENG","URD","URD","ENG","URD","URD"]),
    ("Kal ka function bohot achha raha", ["URD","URD","ENG","URD","URD","URD"]),
    ("Mujhe abhi ek call receive karni hai", ["URD","URD","URD","ENG","ENG","URD","URD"]),
    ("Yaar ye deadline extend ho sakti hai kya", ["URD","URD","ENG","ENG","URD","URD","URD","URD"]),
    ("Mera mood theek nahi hai please mujhe akela chorho", ["URD","URD","URD","URD","URD","ENG","URD","URD","URD"]),
    ("Kal ki party mein kya pehnu main", ["URD","URD","ENG","URD","URD","URD","URD"]),
    ("Mujhe iska cost pata nahi", ["URD","URD","ENG","URD","URD"]),

    ("Yaar internet bohot down chal raha hai poore din se", ["URD","ENG","URD","ENG","URD","URD","URD","URD","URD","URD"]),
    ("Mera interview kal subah 10 baje hai", ["URD","ENG","URD","URD","URD","URD","URD"]),
    ("Kal se mera schedule bohot tight hai", ["URD","URD","URD","ENG","URD","ENG","URD"]),
    ("Yeh khabar sunke mujhe bohot khushi hui", ["URD","URD","URD","URD","URD","URD","URD"]),
    ("Mujhe ye figure out karna hai kaise hoga ye", ["URD","URD","ENG","ENG","URD","URD","URD","URD","URD"]),
    ("Kal wala plan cancel ho gaya hai", ["URD","URD","ENG","ENG","URD","URD","URD"]),
    ("Mera dost bohot supportive hai hamesha", ["URD","URD","URD","ENG","URD","URD"]),
    ("Ye chai thandi ho gayi hai dobara garam karo", ["URD","URD","URD","URD","URD","URD","URD","URD","URD"]),
    ("Kal ka exam postpone hone wala hai shayad", ["URD","URD","ENG","ENG","URD","URD","URD","URD"]),
    ("Mujhe abhi thora rest chahiye honestly", ["URD","URD","URD","ENG","URD","MIX"]),

    ("Yaar office mein network issue chal raha hai", ["URD","ENG","URD","ENG","ENG","URD","URD","URD"]),
    ("Kal se mera weight loss journey start ho raha hai", ["URD","URD","URD","ENG","ENG","ENG","ENG","URD","URD","URD"]),
    ("Mujhe abhi ye khareedna hai urgently", ["URD","URD","URD","URD","URD","ENG"]),
    ("Kal ki flight kis time hai tumhari", ["URD","URD","ENG","URD","ENG","URD","URD"]),
    ("Yaar assignment ka format samajh nahi aya mujhe", ["URD","ENG","URD","ENG","URD","URD","URD","URD"]),
    ("Mera favorite song abhi bhi wahi hai", ["URD","ENG","ENG","URD","URD","URD","URD"]),
    ("Kal raat mujhe neend nahi ayi bilkul", ["URD","URD","URD","URD","URD","URD","URD"]),
    ("Yeh printer kaam nahi kar raha phir se", ["URD","ENG","URD","URD","URD","URD","URD","URD"]),
    ("Mujhe ye jaan kar khushi hui ke tum theek ho", ["URD","URD","URD","URD","URD","URD","URD","URD","URD","URD"]),
    ("Kal wali meeting mein kya discuss hua", ["URD","URD","ENG","URD","URD","ENG","URD"]),
    ("Mera phone screen crack ho gaya hai", ["URD","ENG","ENG","ENG","URD","URD","URD"]),

    ("Yeh season mein bohot rain ho rahi hai", ["URD","ENG","URD","URD","ENG","URD","URD","URD"]),
    ("Mujhe abhi office se leave leni hai", ["URD","URD","ENG","URD","ENG","URD","URD"]),
    ("Kal ka trip cancel ho gaya sad", ["URD","URD","ENG","ENG","URD","URD","ENG"]),
    ("Yaar mujhe kuch samajh nahi araha kya karun", ["URD","URD","URD","URD","URD","URD","URD","URD"]),
    ("Mera pura din waste ho gaya aaj", ["URD","URD","URD","ENG","URD","URD","URD"]),
    ("Kal se online classes start ho rahi hain", ["URD","URD","ENG","ENG","ENG","URD","URD","URD"]),
    ("Yeh shirt size mein chhoti hai mere liye", ["URD","ENG","ENG","ENG","URD","URD","URD","URD"]),
    ("Mujhe abhi ye clear karna hai apne dost se", ["URD","URD","URD","ENG","URD","URD","URD","URD","URD"]),
    ("Kal se raat ka khana bahar khaya karenge", ["URD","URD","URD","URD","URD","URD","URD","URD"]),
    ("Yaar iski call abhi tak nahi ayi", ["URD","URD","ENG","URD","URD","URD","URD"]),
    ("Mera bag ghar pe rah gaya subah", ["URD","ENG","URD","URD","URD","URD","URD"]),

    ("Kal ka homework bohot lamba tha yaar", ["URD","URD","ENG","URD","URD","URD","URD"]),
    ("Mujhe abhi ye samajhna hai kaise kaam karta hai", ["URD","URD","URD","URD","URD","URD","URD","URD","URD"]),
    ("Yeh order abhi tak deliver nahi hua", ["URD","ENG","URD","URD","ENG","URD","URD"]),
    ("Mera pen kho gaya hai kahin", ["URD","ENG","URD","URD","URD","URD"]),
    ("Kal se mera new semester start ho raha hai", ["URD","URD","URD","ENG","ENG","ENG","URD","URD","URD"]),
    ("Yaar tumhara reply itni der se kyun aya", ["URD","URD","ENG","URD","URD","URD","URD","URD"]),
    ("Mujhe abhi ye essay likhna hai final ke liye", ["URD","URD","URD","ENG","URD","URD","ENG","URD","URD"]),
    ("Kal ka din bohot hectic tha yaar sach mein", ["URD","URD","URD","URD","ENG","URD","URD","URD","URD"]),
    ("Yeh design mujhe bohot pasand aya honestly", ["URD","ENG","URD","URD","URD","URD","ENG"]),
    ("Mujhe abhi ye clarify karna hai apne boss se", ["URD","URD","URD","ENG","URD","URD","URD","ENG","URD"]),
    ("Kal se mera diet plan start ho raha hai strictly", ["URD","URD","URD","ENG","ENG","ENG","URD","URD","URD","ENG"]),
    ("Yaar mujhe office jaate hue bohot der ho gayi", ["URD","URD","ENG","URD","URD","URD","URD","URD","URD"]),
    ("Mera dimagh kaam nahi kar raha abhi", ["URD","URD","URD","URD","URD","URD","URD"]),

    ("Kal ka result dekh kar mujhe shock laga", ["URD","URD","ENG","URD","URD","URD","ENG","URD"]),
    ("Yeh bill abhi tak pay nahi hua", ["URD","ENG","URD","URD","ENG","URD","URD"]),
    ("Mujhe abhi ye decide karna hai kis university mein jaun", ["URD","URD","URD","ENG","URD","URD","URD","ENG","URD","URD"]),
    ("Kal se raat ko jaldi sona shuru karenge", ["URD","URD","URD","URD","URD","URD","URD","URD"]),
    ("Yaar iska price bohot zyada hai market mein", ["URD","URD","ENG","URD","URD","URD","ENG","URD"]),
    ("Mera roommate aj wapis a raha hai", ["URD","ENG","URD","URD","URD","URD","URD"]),
    ("Kal ka lecture mujhe samajh hi nahi aya", ["URD","URD","ENG","URD","URD","URD","URD","URD"]),
    ("Yeh event postpone ho gaya hai kal ke liye", ["URD","ENG","ENG","URD","URD","URD","URD","URD","URD"]),
    ("Mujhe abhi ye confirm karna hai apne dost se", ["URD","URD","URD","ENG","URD","URD","URD","URD","URD"]),
    ("Kal se office ka time change ho gaya hai", ["URD","URD","ENG","URD","ENG","ENG","URD","URD","URD"]),
    ("Yaar mujhe ye samajh nahi araha kis se pochun", ["URD","URD","URD","URD","URD","URD","URD","URD","URD"]),
    ("Mera data package khatam ho gaya hai", ["URD","ENG","ENG","URD","URD","URD","URD"]),

    ("Kal ka trip bohot fun tha honestly yaar", ["URD","URD","ENG","URD","ENG","URD","ENG","URD"]),
    ("Yeh news dekh kar mujhe yaqeen nahi hua", ["URD","ENG","URD","URD","URD","URD","URD","URD"]),
    ("Mujhe abhi ye submit karna hai deadline se pehle", ["URD","URD","URD","ENG","URD","URD","ENG","URD","URD"]),
    ("Kal se mera routine bohot disturb ho gaya hai", ["URD","URD","URD","ENG","URD","ENG","URD","URD","URD"]),
    ("Yaar iska number mujhe nahi mil raha", ["URD","URD","ENG","URD","URD","URD","URD"]),
    ("Mera charger kahin gum ho gaya hai", ["URD","ENG","URD","URD","URD","URD","URD"]),
    ("Kal ka session bohot informative tha sach mein", ["URD","URD","ENG","URD","ENG","URD","URD","URD"]),
    ("Yeh update abhi tak nahi aya phone mein", ["URD","ENG","URD","URD","URD","URD","ENG","URD"]),
    ("Mujhe abhi ye request karna hai apne teacher se", ["URD","URD","URD","ENG","URD","URD","URD","ENG","URD"]),
    ("Kal se mera health thora improve ho raha hai", ["URD","URD","URD","ENG","URD","ENG","URD","URD","URD"]),
    ("Yaar iska bill kitna aya hai is dafa", ["URD","URD","ENG","URD","URD","URD","URD","URD"]),
    ("Mera wallet ghar pe hi reh gaya", ["URD","ENG","URD","URD","URD","URD","URD"]),
    ("Kal ka discussion bohot useful tha group mein", ["URD","URD","ENG","URD","ENG","URD","ENG","URD"]),
    ("Yeh photo bohot achi ayi hai tumhari", ["URD","ENG","URD","URD","URD","URD","URD"]),
    ("Mujhe abhi ye explain karna hai sabko", ["URD","URD","URD","ENG","URD","URD","URD"]),

    ("Kal se mera focus bohot better ho gaya hai", ["URD","URD","URD","ENG","URD","ENG","URD","URD","URD"]),
    ("Yaar mujhe ye samajh nahi araha kya karun", ["URD","URD","URD","URD","URD","URD","URD","URD"]),
    ("Mera bag ghar pe rah gaya subah", ["URD","ENG","URD","URD","URD","URD","URD"]),
    ("Kal ka homework bohot lamba tha yaar", ["URD","URD","ENG","URD","URD","URD","URD"]),
    ("Mujhe abhi ye explain karna hai sabko", ["URD", "URD", "URD", "ENG", "URD", "URD", "URD"]),
    ("Kal se mera focus bohot better ho gaya hai", ["URD", "URD", "URD", "ENG", "URD", "ENG", "URD", "URD", "URD"]),
    ("Aaj ka din bohot busy tha, had 3 meetings back to back", ["URD", "URD", "URD", "URD", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG"]),
    ("Yaar seriously I can not even right now, bohot thak gaya hoon", ["URD", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD", "URD"]),
    ("Bhai kal mera presentation hai, still not prepared at all", ["URD", "URD", "URD", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "ENG"]),
    ("Khana kha liya? I was literally waiting for you", ["URD", "URD", "URD", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG"]),
    ("Mera internet connection bohot slow chal raha hai aaj", ["URD", "ENG", "ENG", "URD", "ENG", "URD", "URD", "URD", "URD"]),
    ("Bhai please iss request ko accept karlo jaldi", ["URD", "ENG", "URD", "ENG", "URD", "ENG", "URD", "URD"]),
    ("Is code repository me main branch lock hai", ["URD", "ENG", "ENG", "URD", "ENG", "ENG", "ENG", "URD"]),
    ("Mera server connection sudden lost ho gaya", ["URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Aaj online portal par new assignment upload hui", ["URD", "ENG", "ENG", "URD", "ENG", "ENG", "ENG", "URD"]),
    ("Is product ka user feedback bohot positive aaya", ["URD", "ENG", "URD", "ENG", "ENG", "URD", "ENG", "URD"]),
    ("Bhai project documentation submit kar di hai", ["URD", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),

    ("Kal night match me team performance brilliant thi", ["URD", "ENG", "ENG", "URD", "ENG", "ENG", "ENG", "URD"]),
    ("Higher authority ne official notification issue kiya", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "URD"]),
    ("Mujhe current location pin detail send kar do", ["URD", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Is account key security settings change ho gayi", ["URD", "ENG", "URD", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("New update installation process background me active hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "ENG", "URD"]),
    ("Ye assignment submission deadline kya hai?", ["URD", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Mera laptop screen suddenly freeze ho gaya tha", ["URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Itna heavy traffic tha ke main late ho gaya", ["URD", "ENG", "ENG", "URD", "URD", "URD", "ENG", "URD", "URD"]),
    ("Sir ne kal lecture cancel kar diya tha", ["URD", "URD", "URD", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Aap ne wo email check kiya abhi tak?", ["URD", "URD", "URD", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Mujhe lagta hai ye idea super successful hoga", ["URD", "URD", "URD", "URD", "ENG", "ENG", "ENG", "URD"]),
    ("Kal hamara match complete win tha", ["URD", "URD", "ENG", "ENG", "ENG", "URD"]),
    ("Wahan parking space bilkul nahi thi", ["URD", "ENG", "ENG", "URD", "URD", "URD"]),

    ("Mera mobile battery low hai, main baad me call karta hoon", ["URD", "ENG", "ENG", "ENG", "URD", "URD", "URD", "URD", "ENG", "URD", "URD"]),
    ("Unka budget plan kafi strict hai is baar", ["URD", "ENG", "ENG", "URD", "ENG", "URD", "URD", "URD"]),
    ("Please mujh se direct message me baat karo", ["ENG", "URD", "URD", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Is project me bohot security issues hain", ["URD", "ENG", "URD", "URD", "ENG", "ENG", "URD"]),
    ("Tumhara overall experience kaisa raha wahan?", ["URD", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Mujhe ye new update bilkul pasand nahi aaya", ["URD", "URD", "ENG", "ENG", "URD", "URD", "URD", "URD"]),
    ("Kal hum ne full day enjoy kiya party me", ["URD", "URD", "URD", "ENG", "ENG", "ENG", "URD", "ENG", "URD"]),
    ("Bhai is code me syntax error aa raha hai", ["URD", "URD", "ENG", "URD", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Mera phone notification sound off hai abhi", ["URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Wahan security check timing bohot strict hai", ["URD", "ENG", "ENG", "ENG", "URD", "ENG", "URD"]),
    ("Bohot high priority client message aaya hai", ["URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Is restaurant ki service quality bohot bad thi", ["URD", "ENG", "URD", "ENG", "ENG", "URD", "ENG", "URD"]),
    ("Kal mera online interview schedule hua hai", ["URD", "URD", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Wo hamesha simple steps follower raha hai", ["URD", "URD", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Mujhe ticket confirmation receipt bhej do", ["URD", "ENG", "ENG", "ENG", "URD", "URD"]),

    ("Mera total balance zero ho gaya sudden", ["URD", "ENG", "ENG", "ENG", "URD", "URD", "ENG"]),
    ("Is product ka delivery charger kitna hai?", ["URD", "ENG", "URD", "ENG", "ENG", "URD", "URD"]),
    ("Bhai final output display screen par check karo", ["URD", "ENG", "ENG", "ENG", "ENG", "URD", "ENG", "URD"]),
    ("Kal hamara technical evaluation test hai", ["URD", "URD", "ENG", "ENG", "ENG", "URD"]),
    ("Wo har waqt active online status me rehta hai", ["URD", "URD", "URD", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Is software release note me bugs mentioned hain", ["URD", "ENG", "ENG", "ENG", "URD", "ENG", "ENG", "URD"]),
    ("Mujhe fast internet plan subscribe karna hai", ["URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Wo emergency exit door lock tha", ["URD", "ENG", "ENG", "ENG", "ENG", "URD"]),

    ("State Bank ne new digital currency model introduce kar diya hai", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG","ENG", "URD", "URD", "URD"]),
    ("Government ne petrol ki price me mega increase record kiya", ["ENG", "URD", "ENG", "URD", "ENG", "URD", "ENG", "ENG", "URD", "URD"]),
    ("China aur Pakistan ne AI agreement sign kar liya hai", ["ENG", "URD", "ENG", "URD", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Is company ki total revenue last month double ho gayi", ["URD", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Federal Cabinet ne new tax policy approve kar di hai", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Stock market today record high level par close hui hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "ENG", "URD", "URD"]),
    ("Gold price worldwide seven weeks high level par pohench gayi", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("IT export me is saal massive growth dekhne ko mili", ["ENG", "ENG", "URD", "URD", "URD", "ENG", "ENG", "URD", "URD", "URD"]),
    ("State Bank official press release issue karne wala hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("New startup platform ne seed funding round complete kiya", ["ENG", "ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("Pakistan ne second test match 8 wickets se jeet liya", ["ENG", "URD", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("PCB ne player par 2 years ki ban announce kar di", ["ENG", "URD", "ENG", "URD", "ENG", "ENG", "URD", "ENG", "ENG", "URD", "URD"]),
    ("Green shirts next week final match play kare gi", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),

    ("Captain ne match presentation me team performance ko cherish kiya", ["ENG", "URD", "ENG", "ENG", "URD", "ENG", "ENG", "URD", "ENG", "URD"]),
    ("T20 World Cup squad announce ho gaya hai aaj", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD", "URD"]),
    ("Young player ne debut match me century score ki", ["ENG", "ENG", "URD", "ENG", "ENG", "URD", "ENG", "ENG", "URD"]),
    ("Olympic final match rain ki wajah se delay ho gaya", ["ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD", "ENG", "URD", "URD"]),
    ("Fast bowler ne crucial moment par wicket break ki", ["ENG", "ENG", "URD", "ENG", "ENG", "URD", "ENG", "ENG", "URD"]),
    ("Meteorological department ne heavy rain alert issue kar diya", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Highway road heavy flood waters ki wajah se closed hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD", "ENG", "URD"]),
    ("Islamabad Police ne traffic city management order publish kiya", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("City administration ne emergency control room setup kar diya hai", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Severe heatwave alert nationwide public issue ho gaya", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),

    ("PMDC ne entrance test MDCAT date reschedule kar di", ["ENG", "URD", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Matric board results today online portal par announce honge", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "ENG", "URD"]),
    ("Education ministry ne free digital training courses offer kiye", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("University administration ne semester exams offline mode me rakhe", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Higher Education Commission ne new scholarship policy approve ki", ["ENG", "ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("Viral video challenge social media platforms par trend kar raha hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "ENG", "URD", "URD", "URD"]),
    ("Cyber security team ne massive online fraud catch kiya", ["ENG", "ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD"]),

    ("Artificial intelligence tool human writers ko beat kar raha hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "ENG", "URD", "URD", "URD"]),
    ("Space agency ne new satellite orbit me launch kar diya", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "URD", "ENG", "URD", "URD"]),
    ("Popular messaging application ne thousands of accounts block kiye", ["ENG", "ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("New movie release six days me box office benchmark set kar chuki hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD","URD", "URD"]),
    ("High Court ne case key hearing next Monday tak adjourn ki", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "ENG","URD", "ENG", "URD"]),
    ("Police team ne suspect vehicle trace kar ke impound kar liya", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "URD", "URD", "ENG", "URD", "URD"]),
    ("Anti-corruption court ne bail plea reject kar di", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Investigation officer ne formal charge sheet submit kar di hai", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Cyber crime unit ne hacker group key network trace kiye", ["ENG", "ENG", "ENG", "URD", "ENG", "ENG", "URD", "ENG", "ENG", "URD"]),
    ("Supreme Court landmark judgment detail publish kar di hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),

    ("AI technology ne content writing ko redefine kar diya", ["ENG", "ENG", "URD", "ENG", "ENG", "URD","ENG", "URD", "URD"]),
    ("Nvidia ne latest AI chip reveal kar di hai", ["ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Cyber attack se company ka database corrupt ho gaya", ["ENG", "ENG", "URD", "ENG", "URD", "ENG", "ENG", "URD", "URD"]),
    ("New software update installation timing late schedule hui hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Social media app par privacy policy update aa gayi", ["ENG", "ENG", "ENG", "URD", "ENG", "ENG", "ENG", "URD", "URD"]),

    ("Inflation rate record high mark par reach kar chuka hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "ENG", "URD", "URD", "URD"]),
    ("Stock exchange index drop hone se investors worried hain", ["ENG", "ENG", "ENG", "ENG", "URD", "URD", "ENG", "ENG", "URD"]),
    ("Banking sector ne online fraud recovery plan issue kiya", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("Export rate high hone se national economy boost hogi", ["ENG", "ENG", "ENG", "URD", "URD", "ENG", "ENG", "ENG", "URD"]),
    ("Fuel price decrease karne par public demand increase hui", ["ENG", "ENG", "ENG", "URD", "URD", "ENG", "ENG", "ENG", "URD"]),

    ("Championship match schedule rain alert ki wajah se shift hua", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD", "ENG", "URD"]),
    ("Head coach ne team lineup final decision announcement kar diya", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Cricket board ne player fitness test result publish kar diya", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Stadium management ne ticketing system digitalize kar diya hai", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Young bowler ne hat trick performance record ki aaj", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Heavy smog alert ki wajah se schools close rahenge", ["ENG", "ENG", "ENG", "URD", "URD", "URD", "ENG", "ENG", "URD"]),

    ("Traffic police ne highway safety camera system activate kiya", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("Metro bus service route emergency work ki waja se divert hua", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD", "ENG", "URD"]),
    ("Power shutdown notice city electric supplier ne issue kiya", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "ENG", "URD"]),
    ("Clean green drive project municipality team start karne wali hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Board exams date sheet change hone se students relax hue", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "ENG", "ENG", "URD"]),
    ("University merit list upload hone me delay expected hai", ["ENG", "ENG", "ENG", "ENG", "URD", "URD", "ENG", "ENG", "URD"]),
    ("Scholarship application process complete online system par available hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "ENG", "URD"]),
    ("Higher Education Department ne mandatory attendance rule update kiya", ["ENG", "ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("Higher entry test pattern change karne ki proposal review hui", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "ENG", "ENG", "URD"]),
    ("Documentary trailer drop hotay hi viral trend ban gaya", ["ENG", "ENG", "ENG", "URD", "URD", "ENG", "ENG", "URD", "URD"]),
    ("New movie box office opening weekend target complete kar chuki", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Content creator ne streaming channel subscriber milestone achieve kiya", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("Celebrity interview clip overall social media trend ban gayi", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),

    ("New model training process complete ho gaya hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Is dataset me word level tokenization apply karni padegi", ["URD", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Server response delay ki waja se connection timeout hua", ["ENG", "ENG", "ENG", "URD", "URD", "URD", "ENG", "ENG", "URD"]),
    ("Local machine par GPU memory limit exceed kar gayi", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Pro player ne clutch round win kar ke match bacha liya", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "URD", "URD", "ENG", "URD", "URD"]),
    ("Game update ke baad ping issue solve nahi hua", ["ENG", "ENG", "URD", "URD", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Team captain ne team fight decision change kar diya", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("New season battle pass discount rate par available hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "ENG", "URD"]),

    ("Client meeting link update karna bhool gaya main", ["ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD", "URD"]),
    ("Is freelance project ki milestone deadline verify kar lo", ["URD", "ENG", "ENG", "URD", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Team manager ne overall project progress admire ki", ["ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("Portfolio website UI design fast response de raha hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Airport security check queue bohot long thi aaj", ["ENG", "ENG", "ENG", "ENG", "URD", "ENG", "URD", "URD"]),
    ("Express train timing change hone se schedule disturb hua", ["ENG", "ENG", "ENG", "ENG", "URD", "URD", "ENG", "ENG", "URD"]),
    ("City bypass highway track maintenance under process hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("Electric bus route launching event delay ho gaya hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),

    ("Online shopping app mega sale discount offer de rahi hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Cinema hall booking system crash hone se public annoyed thi", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "ENG", "ENG", "URD"]),
    ("New smartphone camera sensor performance top class hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("Wireless earbuds battery life expect se double mili", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "ENG", "URD"]),
    ("Semester final grade report portal par display ho gayi", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "ENG", "URD", "URD"]),
    ("Library card extension request form submit kar diya hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Campus canteen management quality review meeting schedule hui", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("Research paper abstract submission date delay ho sakti hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("grade report portal par display" , ["ENG","ENG" , "ENG", "URD", "URD"]),
    ("Mene ajj boht acha Pasta Khaya" , ["URD", "URD", "URD","URD","ENG","URD"]),
    ("Ajj Ten Class Ka Result Tha" , ["URD","ENG","ENG","URD","ENG","URD"]),
    ("Sit Down Beta" , ["ENG", "ENG", "URD"]),

    ("Smart watch notification alert delay se aa raha", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Is presentation slide ka visual layout change karo", ["URD", "ENG", "ENG", "URD", "ENG", "ENG", "ENG", "URD"]),
    ("Flight booking status online modify kar diya hai", ["ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD", "URD"]),
    ("Is web application me security patch add update", ["URD", "ENG", "ENG", "URD", "ENG", "ENG", "ENG", "ENG"]),
    ("Cyber attack warning clear detail print out nikalo", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD"]),
    ("Online order delivery driver address query kar raha", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Digital marketing strategy plan approval complete ho gayi", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Software bug reporting tool link share kar do", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),
    ("Mera password reset link inbox me receive hua", ["URD", "ENG", "ENG", "ENG", "ENG", "URD", "ENG", "URD"]),
    ("Higher entry test scorecard update publish kar di", ["ENG", "ENG", "ENG", "ENG", "ENG", "ENG", "URD", "URD"]),

]

# 2. Build flat dataframe
rows = []
for sentence, labels in sentences_with_labels:
    words = sentence.split()
    if len(words) != len(labels):
        raise ValueError(f"Mismatch in: {sentence} ({len(words)} words vs {len(labels)} labels)")
    for word, label in zip(words, labels):
        rows.append({"sentence": sentence, "word": word, "label": label})

df = pd.DataFrame(rows)
df.to_csv("dataset.csv", index=False, encoding="utf-8")

print(f"Total sentences: {len(sentences_with_labels)}")
print(f"Total word entries: {len(df)}")
print("Label distribution:")
print(df.label.value_counts())

Total sentences: 334
Total word entries: 2684
Label distribution:
label
URD    1510
ENG    1170
MIX       4
Name: count, dtype: int64


In [5]:
import pandas as pd

# 1. Flatten sentences into word-level entries
rows = []
for item in sentences_with_labels:
    # Handle both tuple format (sentence, [labels]) and dictionary format {'sentence': ..., 'words': ..., 'labels': ...}
    if isinstance(item, tuple):
        sentence, labels = item
        words = sentence.split()
    elif isinstance(item, dict):
        sentence = item['sentence']
        words = item.get('words', sentence.split())
        labels = item['labels']

    # Zip words and labels into flat rows
    for word, label in zip(words, labels):
        rows.append({
            'sentence': sentence,
            'word': word,
            'label': label
        })

# 2. Create DataFrame and export to CSV
df = pd.DataFrame(rows)
df.to_csv('dataset.csv', index=False, encoding='utf-8')

# 3. Print validation summary
print(f"✅ CSV Created Successfully!")
print(f"Total Sentences: {df['sentence'].nunique()}")
print(f"Total Labeled Words: {len(df)}")
print("\nLabel Counts:")
print(df['label'].value_counts())

✅ CSV Created Successfully!
Total Sentences: 330
Total Labeled Words: 2684

Label Counts:
label
URD    1510
ENG    1170
MIX       4
Name: count, dtype: int64


In [9]:
!pip install transformers torch datasets seqeval
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
# Load your dataset
df = pd.read_csv('dataset.csv')
# Create label mapping
label2id = {'URD': 0, 'ENG': 1, 'MIX': 2}
id2label = {0: 'URD', 1: 'ENG', 2: 'MIX'}
# Group by sentence
sentences = df.groupby('sentence').apply(
lambda x: {'words': x['word'].tolist(), 'labels': x['label'].tolist()}
).tolist()
# Split into train and test
train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)
print(f'Training sentences: {len(train_data)}')
print(f'Testing sentences: {len(test_data)}')

Training sentences: 264
Testing sentences: 66


/tmp/ipykernel_621/3405380065.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sentences = df.groupby('sentence').apply(


In [17]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from datasets import Dataset


# -------------------------------
# 1. Load XLM-RoBERTa model
# -------------------------------

model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)


# -------------------------------
# 2. Tokenize and align labels
# -------------------------------

def tokenize_and_align_labels(examples):

    tokenized = tokenizer(
        examples["words"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["labels"]):

        word_ids = tokenized.word_ids(batch_index=i)

        label_ids = []
        prev_word = None

        for word_id in word_ids:

            # Special tokens
            if word_id is None:
                label_ids.append(-100)

            # First token of a word
            elif word_id != prev_word:
                label_ids.append(
                    label2id[label[word_id]]
                )

            # Remaining subword tokens
            else:
                label_ids.append(-100)

            prev_word = word_id

        labels.append(label_ids)

    tokenized["labels"] = labels

    return tokenized


# -------------------------------
# 3. Convert data to HF Dataset
# -------------------------------

def to_hf_dataset(data):

    return Dataset.from_dict({
        "words": [d["words"] for d in data],
        "labels": [d["labels"] for d in data]
    })


# Create datasets
train_dataset = to_hf_dataset(train_data)
test_dataset = to_hf_dataset(test_data)


# -------------------------------
# 4. Apply tokenization
# -------------------------------

train_ds = train_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=train_dataset.column_names
)

test_ds = test_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=test_dataset.column_names
)


# -------------------------------
# 5. Training settings
# -------------------------------

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=10,

    load_best_model_at_end=True,

    report_to="none"
)


# -------------------------------
# 6. Data collator
# -------------------------------

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)


# -------------------------------
# 7. Create Trainer
# -------------------------------

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_ds,
    eval_dataset=test_ds,

    processing_class=tokenizer,

    data_collator=data_collator
)


# -------------------------------
# 8. Start training
# -------------------------------

print("Starting training...")

trainer.train()

print("Training complete!")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/264 [00:00<?, ? examples/s]

Map:   0%|          | 0/66 [00:00<?, ? examples/s]

Starting training...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


In [18]:
from huggingface_hub import notebook_login

# Step 1: Login to Hugging Face
notebook_login()

In [19]:
repo_name = "code-switching-codesaviours-si26-saima"

model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print(f"Model published at: https://huggingface.co/Saima109/{repo_name}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...381l5b0/model.safetensors:   0%|          | 13.6kB / 1.11GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mppftaq84_/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

Model published at: https://huggingface.co/Saima109/code-switching-codesaviours-si26-saima
